# FYP Phase 2 — Restormer fine-tuning, any blur level (5–10)

Change **only `LEVEL`** in the first cell, then *Runtime → Run all*.

Everything is read from / written to the per-level Drive layout:

```
FinalYearProject-Phase2/
├── pretrained/single_image_defocus_deblurring.pth
└── level_XX/
    ├── dataset/      HDC2021_stepN.zip
    ├── checkpoints/  levelN_best.pth, levelN_last.pth
    ├── results/      split, metrics CSVs, OCR outputs, figures/, restored/
    ├── logs/         training_history.json, train_log.txt
    └── manifest.json
```

In [ ]:
# ============================================================
# CONFIG — the only thing you change between runs
# ============================================================
LEVEL     = 6              # 5, 6, 7, 8, 9 or 10
INIT_FROM = 'pretrained'   # 'pretrained' -> start from original Restormer weights (same as Level 5, fair comparison)
                           # 'previous'   -> start from the best checkpoint of LEVEL-1 (curriculum fine-tuning)
RESUME    = True           # continue from levelN_last.pth if it exists

from google.colab import drive
drive.mount('/content/drive')

import os, glob, shutil

PHASE2     = '/content/drive/MyDrive/FinalYearProject-Phase2'
PRETRAINED = f'{PHASE2}/pretrained/single_image_defocus_deblurring.pth'

LEVEL_DIR = f'{PHASE2}/level_{LEVEL:02d}'
DATA_DIR  = f'{LEVEL_DIR}/dataset'
CKPT_DIR  = f'{LEVEL_DIR}/checkpoints'
LOG_DIR   = f'{LEVEL_DIR}/logs'
OUT       = f'{LEVEL_DIR}/results'

# Level 5 was trained before the reorganisation -> its outputs live one folder deeper
legacy_out = f'{OUT}/level{LEVEL}_results'
if os.path.isdir(legacy_out):
    OUT = legacy_out

FIG_DIR  = f'{OUT}/figures'
REST_DIR = f'{OUT}/restored'
LOCAL    = f'/content/step{LEVEL}_data'     # fast local copy (Drive I/O is slow)

assert os.path.isdir(PHASE2), f'Folder not found: {PHASE2}'
assert os.path.isdir(LEVEL_DIR), f'Level folder not found: {LEVEL_DIR}'
assert os.path.exists(PRETRAINED), f'Pretrained weights not found: {PRETRAINED}'

for d in [DATA_DIR, CKPT_DIR, LOG_DIR, OUT, FIG_DIR, REST_DIR, LOCAL]:
    os.makedirs(d, exist_ok=True)


def level_ckpt(level, kind):
    """Path of levelN_best/last.pth for any level (also accepts the old 'checkpoints__' prefix)."""
    folder = f'{PHASE2}/level_{level:02d}/checkpoints'
    clean_name = f'{folder}/level{level}_{kind}.pth'
    old_name = f'{folder}/checkpoints__level{level}_{kind}.pth'
    if os.path.exists(clean_name):
        return clean_name
    if os.path.exists(old_name):
        return old_name
    return clean_name


def level_result_file(level, name):
    """Find a results file for any level (handles Level 5's nested levelN_results folder)."""
    base = f'{PHASE2}/level_{level:02d}/results'
    for p in [f'{base}/level{level}_results/{name}', f'{base}/{name}']:
        if os.path.exists(p):
            return p
    return None


BEST = level_ckpt(LEVEL, 'best')
LAST = level_ckpt(LEVEL, 'last')

# ---- locate the dataset: the expected zip, any zip, or an already-extracted folder ----
ZIP_PATH = f'{DATA_DIR}/HDC2021_step{LEVEL}.zip'
if not os.path.exists(ZIP_PATH):
    any_zip = sorted(glob.glob(f'{DATA_DIR}/*.zip'))
    if len(any_zip) > 0:
        ZIP_PATH = any_zip[0]

DRIVE_DIR = None
cam_dirs = [d for d in glob.glob(f'{DATA_DIR}/**/CAM01', recursive=True) if '__MACOSX' not in d]
if len(cam_dirs) > 0:
    DRIVE_DIR = DATA_DIR

# ---- free local disk from other levels' extracted data ----
for d in glob.glob('/content/step*_data'):
    if d != LOCAL:
        shutil.rmtree(d, ignore_errors=True)
        print('Removed old local copy:', d)

print(f'LEVEL {LEVEL}  (init from: {INIT_FROM})')
print('Dataset folder :', os.listdir(DATA_DIR))
print('Zip            :', os.path.exists(ZIP_PATH), '->', ZIP_PATH)
print('Extracted copy :', DRIVE_DIR)
print('Pretrained     :', PRETRAINED)
print('Checkpoints    :', BEST, '|', LAST)
print('Outputs        :', OUT)

In [ ]:
!pip install -q einops tifffile scikit-image easyocr rapidfuzz

import re, json, random, time
from datetime import datetime
import numpy as np
import pandas as pd
import cv2, tifffile
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from rapidfuzz.distance import Levenshtein

assert torch.cuda.is_available(), 'No GPU - Runtime > Change runtime type > T4 GPU'
DEVICE = 'cuda'
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print('PyTorch:', torch.__version__)
print('GPU    :', torch.cuda.get_device_name(0))
print('VRAM   :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
already = [d for d in glob.glob(f'{LOCAL}/**/CAM01', recursive=True) if '__MACOSX' not in d]

if len(already) > 0:
    print('Already extracted:', already)
elif os.path.exists(ZIP_PATH):
    print(f'Copying {os.path.basename(ZIP_PATH)} from Drive (~2.5 GB, 2-4 min)...')
    shutil.copy(ZIP_PATH, f'/content/step{LEVEL}.zip')
    print('Unzipping...')
    !unzip -q -o "/content/step{LEVEL}.zip" -d "{LOCAL}"
    os.remove(f'/content/step{LEVEL}.zip')
elif DRIVE_DIR is not None:
    print('Copying extracted folder from Drive...')
    shutil.copytree(DRIVE_DIR, f'{LOCAL}/step{LEVEL}', dirs_exist_ok=True)
else:
    raise FileNotFoundError(f'Upload HDC2021_step{LEVEL}.zip to {DATA_DIR}')

for d in sorted(glob.glob(f'{LOCAL}/**/CAM0*', recursive=True)):
    if '__MACOSX' not in d:
        print(d, '->', len(glob.glob(d + '/*sample_*.tif')), 'images')

In [ ]:
SAMPLE_RE = re.compile(r'^focusStep_' + str(LEVEL) + r'_(timesR|verdanaRef)_size_\d+_sample_(\d{4})\.tif$')

def read_raw(path):
    img = tifffile.imread(path).astype(np.float32)
    if img.ndim == 3:
        img = img.mean(axis=-1)
    return img

def normalise(img):
    bg = np.percentile(img, 99.5)
    if bg <= 0:
        bg = img.max() + 1e-6
    return np.clip(img / bg, 0.0, 1.0)

pairs = []
for root, dirs, files in os.walk(LOCAL):
    if '__MACOSX' in root:
        continue
    if os.path.basename(root) != 'CAM01':
        continue
    cam2_dir = os.path.join(os.path.dirname(root), 'CAM02')
    if not os.path.isdir(cam2_dir):
        print('WARNING no CAM02 next to', root)
        continue
    cam1_set = set(f for f in files if SAMPLE_RE.match(f))
    cam2_set = set(f for f in os.listdir(cam2_dir) if SAMPLE_RE.match(f))
    if cam1_set != cam2_set:
        print('WARNING filename mismatch in', root)
    for f in sorted(cam1_set & cam2_set):
        m = SAMPLE_RE.match(f)
        pairs.append({
            'id':     f'{m.group(1)}_{m.group(2)}',
            'font':   m.group(1),
            'sample': int(m.group(2)),
            'cam01':  os.path.join(root, f),
            'cam02':  os.path.join(cam2_dir, f),
            'txt':    os.path.join(root, f.replace('.tif', '.txt')),
        })

assert len(pairs) > 0, f'No focusStep_{LEVEL}_*.tif pairs found - check the zip is really step {LEVEL}'
print('Total pairs:', len(pairs))
print(pd.Series([p['font'] for p in pairs]).value_counts().to_string())

# HDC 2021 convention: CAM01 = focused (ground truth), CAM02 = defocused (input)
SHARP_KEY, BLUR_KEY = 'cam01', 'cam02'

def edge_energy(img):
    # Smooth first so sensor noise is NOT counted as sharpness
    sm = cv2.GaussianBlur(img, (0, 0), 2.0)
    gy, gx = np.gradient(sm)
    return float(np.mean(gx ** 2 + gy ** 2))

e_sharp, e_blur = [], []
for p in random.Random(SEED).sample(pairs, min(10, len(pairs))):
    e_sharp.append(edge_energy(normalise(read_raw(p[SHARP_KEY]))))
    e_blur.append(edge_energy(normalise(read_raw(p[BLUR_KEY]))))
print(f'Edge energy  {SHARP_KEY.upper()} (GT): {np.mean(e_sharp):.6f}   {BLUR_KEY.upper()} (input): {np.mean(e_blur):.6f}')
assert np.mean(e_sharp) > np.mean(e_blur), 'GT camera is not sharper than input camera - STOP and check'

# Visual confirmation
p = pairs[0]
s = normalise(read_raw(p[SHARP_KEY]))
b = normalise(read_raw(p[BLUR_KEY]))
H, W = s.shape
fig, ax = plt.subplots(1, 2, figsize=(16, 3))
ax[0].imshow(b[H // 2 - 200:H // 2 + 200, W // 2 - 600:W // 2 + 600], cmap='gray', vmin=0, vmax=1)
ax[0].set_title(f'LEVEL {LEVEL} INPUT ({BLUR_KEY.upper()}) - should look BLURRED')
ax[1].imshow(s[H // 2 - 200:H // 2 + 200, W // 2 - 600:W // 2 + 600], cmap='gray', vmin=0, vmax=1)
ax[1].set_title(f'GROUND TRUTH ({SHARP_KEY.upper()}) - should look SHARP')
for a in ax:
    a.axis('off')
plt.show()

In [ ]:
SPLIT_FILE = f'{OUT}/split.json'
N_TEST_PER_FONT = 20
N_VAL_PER_FONT  = 5

if os.path.exists(SPLIT_FILE):
    split = json.load(open(SPLIT_FILE))
    print('Loaded existing split')
else:
    rng = random.Random(SEED)
    split = {'train': [], 'val': [], 'test': []}
    for font in ['timesR', 'verdanaRef']:
        ids = sorted(p['id'] for p in pairs if p['font'] == font)
        assert len(ids) > N_TEST_PER_FONT + N_VAL_PER_FONT, f'Too few {font} images: {len(ids)}'
        rng.shuffle(ids)
        split['test']  += ids[:N_TEST_PER_FONT]
        split['val']   += ids[N_TEST_PER_FONT:N_TEST_PER_FONT + N_VAL_PER_FONT]
        split['train'] += ids[N_TEST_PER_FONT + N_VAL_PER_FONT:]
    json.dump(split, open(SPLIT_FILE, 'w'), indent=2)

tr, va, te = set(split['train']), set(split['val']), set(split['test'])
assert len(tr & va) == 0 and len(tr & te) == 0 and len(va & te) == 0, 'LEAKAGE'
all_ids = set(p['id'] for p in pairs)
assert (tr | va | te) <= all_ids, 'split.json refers to images not in this dataset'
print(f"Train {len(tr)} | Val {len(va)} | Test {len(te)} | disjoint: True")

In [ ]:
MARGIN = 48                                 # pixels trimmed from every border after warping
SIGMA_CANDIDATES = list(range(5, 105, 5))   # Level 5 was ~25 px; higher levels blur more

def blur_for_matching(img, sigma):
    return cv2.GaussianBlur(img, (0, 0), sigma)

def estimate_blur_sigma(s, b):
    H, W = s.shape
    cy, cx = H // 2, W // 2
    R = 60
    region = b[cy - 300 - R:cy + 300 + R, cx - 800 - R:cx + 800 + R]
    best_sig, best_ncc = 0, -1.0
    for sig in SIGMA_CANDIDATES:
        tmpl = blur_for_matching(s, sig)[cy - 300:cy + 300, cx - 800:cx + 800]
        ncc = cv2.matchTemplate(region, tmpl, cv2.TM_CCOEFF_NORMED).max()
        if ncc > best_ncc:
            best_sig, best_ncc = sig, ncc
    return best_sig

def match_points(s, b, sigma, min_ncc=0.8, ph=160, pw=320, R=60, step=80):
    sb = blur_for_matching(s, sigma)
    H, W = s.shape
    src, dst = [], []
    for y in range(R + ph // 2, H - R - ph // 2, step):
        for x in range(R + pw // 2, W - R - pw // 2, step):
            if s[y - ph // 2:y + ph // 2, x - pw // 2:x + pw // 2].std() < 0.08:
                continue
            tmpl = sb[y - ph // 2:y + ph // 2, x - pw // 2:x + pw // 2]
            region = b[y - ph // 2 - R:y + ph // 2 + R, x - pw // 2 - R:x + pw // 2 + R]
            r = cv2.matchTemplate(region, tmpl, cv2.TM_CCOEFF_NORMED)
            _, mx, _, loc = cv2.minMaxLoc(r)
            if mx < min_ncc:
                continue
            src.append((x, y))
            dst.append((x + loc[0] - R, y + loc[1] - R))
    return np.float32(src).reshape(-1, 2), np.float32(dst).reshape(-1, 2)

def warp_gt(s, M):
    H, W = s.shape
    return cv2.warpAffine(s, M, (W, H), flags=cv2.INTER_CUBIC | cv2.WARP_INVERSE_MAP, borderMode=cv2.BORDER_REPLICATE)

def align_score(s, b, sigma):
    # correlation between blurred GT and real input over the text area (higher = better aligned)
    sb = blur_for_matching(s, sigma)
    m = cv2.dilate((cv2.GaussianBlur(b, (0, 0), 5) < 0.9).astype(np.uint8), np.ones((61, 61), np.uint8)) > 0
    m[:80] = False
    m[-80:] = False
    m[:, :80] = False
    m[:, -80:] = False
    return float(np.corrcoef(sb[m], b[m])[0, 1])

# 1) Load and normalise everything
raw = {}
gt_text = {}
for p in tqdm(pairs, desc='Loading'):
    s = normalise(read_raw(p[SHARP_KEY]))
    b = normalise(read_raw(p[BLUR_KEY]))
    assert s.shape == b.shape, p['id']
    raw[p['id']] = (b, s)
    if os.path.exists(p['txt']):
        gt_text[p['id']] = open(p['txt'], encoding='utf-8', errors='ignore').read()
    else:
        gt_text[p['id']] = ''

# 2) Fit ONE global affine (camera geometry is fixed) using TRAINING images only
fit_ids = random.Random(SEED).sample(split['train'], min(12, len(split['train'])))
BLUR_SIGMA = int(np.median([estimate_blur_sigma(raw[i][1], raw[i][0]) for i in fit_ids[:4]]))
if BLUR_SIGMA >= SIGMA_CANDIDATES[-1]:
    print(f'WARNING: blur sigma hit the search limit ({BLUR_SIGMA}px) - extend SIGMA_CANDIDATES')

M = None
for min_ncc in [0.8, 0.7, 0.6]:      # heavier blur -> weaker matches, so relax gradually
    all_src, all_dst = [], []
    for i in tqdm(fit_ids, desc=f'Matching (NCC >= {min_ncc})'):
        src, dst = match_points(raw[i][1], raw[i][0], BLUR_SIGMA, min_ncc=min_ncc)
        all_src.append(src)
        all_dst.append(dst)
    all_src = np.concatenate(all_src)
    all_dst = np.concatenate(all_dst)
    if len(all_src) >= 50:
        # M maps INPUT pixel coords -> GT pixel coords (used with WARP_INVERSE_MAP)
        M, inl = cv2.estimateAffine2D(all_dst, all_src, ransacReprojThreshold=3.0, maxIters=5000)
        if M is not None:
            print(f'Blur sigma ~{BLUR_SIGMA}px | {len(all_src)} matched patches | {int(inl.sum())} RANSAC inliers (NCC >= {min_ncc})')
            break
    print(f'Only {len(all_src)} matches at NCC >= {min_ncc}, relaxing...')

if M is None:
    prev_affine = level_result_file(LEVEL - 1, 'registration_affine.npy')
    assert prev_affine is not None, 'Registration failed and no previous-level affine to fall back on'
    M = np.load(prev_affine)
    print('WARNING: registration failed - using previous level affine:', prev_affine)

print('Affine (input -> GT):\n', np.round(M, 4))
np.save(f'{OUT}/registration_affine.npy', M)

# 3) Warp every GT into the input frame, trim borders, check alignment on every pair
data, checks = {}, []
for pid in tqdm(raw, desc='Registering'):
    b, s = raw[pid]
    w = np.clip(warp_gt(s, M), 0, 1)
    before = align_score(s, b, BLUR_SIGMA)
    after = align_score(w, b, BLUR_SIGMA)
    b_c = b[MARGIN:-MARGIN, MARGIN:-MARGIN]
    w_c = w[MARGIN:-MARGIN, MARGIN:-MARGIN]
    data[pid] = (b_c.astype(np.float16), w_c.astype(np.float16))
    checks.append({'id': pid, 'ncc_before': before, 'ncc_after': after})
del raw

chk = pd.DataFrame(checks)
chk.to_csv(f'{OUT}/pairing_checks.csv', index=False)
print(f"Alignment NCC  before: mean {chk.ncc_before.mean():.3f} | after: mean {chk.ncc_after.mean():.3f}, min {chk.ncc_after.min():.3f}")
if chk.ncc_after.mean() < chk.ncc_before.mean():
    print('WARNING: alignment got WORSE after warping - inspect pairing_checks.csv before training')
print('Pairs with NCC_after < 0.7 (check these):', chk[chk.ncc_after < 0.7].id.tolist())
print('Image size after trim:', data[pairs[0]['id']][0].shape)
print(f'RAM used by images: {sum(v[0].nbytes + v[1].nbytes for v in data.values()) / 1e9:.2f} GB')

In [ ]:
import importlib.util

RESTORMER_DIR = '/content/Restormer'
if not os.path.exists(RESTORMER_DIR):
    !git clone -q https://github.com/swz30/Restormer.git {RESTORMER_DIR}

# Load only the architecture file (avoids basicsr import problems)
spec = importlib.util.spec_from_file_location('restormer_arch', f'{RESTORMER_DIR}/basicsr/models/archs/restormer_arch.py')
arch = importlib.util.module_from_spec(spec)
spec.loader.exec_module(arch)

print('Pretrained weights:', PRETRAINED, '|', round(os.path.getsize(PRETRAINED) / 1e6, 1), 'MB')

def build_model(weights=PRETRAINED):
    m = arch.Restormer(inp_channels=3, out_channels=3, dim=48, num_blocks=[4, 6, 6, 8],
                       num_refinement_blocks=4, heads=[1, 2, 4, 8], ffn_expansion_factor=2.66,
                       bias=False, LayerNorm_type='WithBias', dual_pixel_task=False)
    ck = torch.load(weights, map_location='cpu')
    if 'params' in ck:
        ck = ck['params']
    m.load_state_dict(ck, strict=True)
    return m.to(DEVICE)

# Starting point for fine-tuning
if INIT_FROM == 'previous':
    INIT_WEIGHTS = level_ckpt(LEVEL - 1, 'best')
    assert os.path.exists(INIT_WEIGHTS), f'No Level {LEVEL - 1} best checkpoint at {INIT_WEIGHTS}'
else:
    INIT_WEIGHTS = PRETRAINED
print('Fine-tuning starts from:', INIT_WEIGHTS)

_m = build_model()
print('Total params:', round(sum(p.numel() for p in _m.parameters()) / 1e6, 2), 'M')
del _m
torch.cuda.empty_cache()

In [ ]:
USE_AMP = True
TILE = 768
if BLUR_SIGMA <= 50:
    OVERLAP = 192
else:
    OVERLAP = 256      # wider blur -> wider seam blending
print(f'Inference tiles: {TILE}px, overlap {OVERLAP}px (blur sigma ~{BLUR_SIGMA}px)')

@torch.no_grad()
def restore(model, img2d, tile=TILE, overlap=OVERLAP):
    model.eval()
    H, W = img2d.shape
    ph = max(tile, int(np.ceil(H / 8) * 8)) - H
    pw = max(tile, int(np.ceil(W / 8) * 8)) - W
    x = torch.from_numpy(img2d.astype(np.float32))[None, None].repeat(1, 3, 1, 1).to(DEVICE)
    x = torch.nn.functional.pad(x, (0, pw, 0, ph), mode='reflect')
    Hp, Wp = x.shape[-2:]
    stride = tile - overlap

    def starts(n):
        if n <= tile:
            return [0]
        s = list(range(0, n - tile, stride))
        s.append(n - tile)
        return s

    # feathered weights so tile seams blend smoothly
    ramp = torch.ones(tile, device=DEVICE)
    edge = torch.linspace(0.05, 1.0, overlap // 2, device=DEVICE)
    ramp[:overlap // 2] = edge
    ramp[-(overlap // 2):] = edge.flip(0)
    win = (ramp[:, None] * ramp[None, :])[None, None]

    out = torch.zeros_like(x)
    wgt = torch.zeros((1, 1, Hp, Wp), device=DEVICE)
    for y in starts(Hp):
        for x0 in starts(Wp):
            patch = x[:, :, y:y + tile, x0:x0 + tile]
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
                r = model(patch)
            out[:, :, y:y + tile, x0:x0 + tile] += r.float() * win
            wgt[:, :, y:y + tile, x0:x0 + tile] += win
    res = (out / wgt)[:, :, :H, :W].mean(1)[0].clamp(0, 1)
    return res.cpu().numpy()

def img_metrics(pred, gt):
    pred = pred.astype(np.float32)
    gt = gt.astype(np.float32)
    return (peak_signal_noise_ratio(gt, pred, data_range=1.0),
            structural_similarity(gt, pred, data_range=1.0))

def save_png16(img, path):
    cv2.imwrite(path, (np.clip(img.astype(np.float32), 0, 1) * 65535).round().astype(np.uint16))

print('Helpers ready')

In [ ]:
res = {'Blurred input': {}, 'Pretrained Restormer': {}, 'Fine-tuned Restormer': {}}
restored = {'Blurred input': {}, 'Pretrained Restormer': {}, 'Fine-tuned Restormer': {}}
os.makedirs(f'{REST_DIR}/pretrained', exist_ok=True)

# Baseline is ALWAYS the original pretrained Restormer (same reference for every level)
model_pre = build_model(PRETRAINED)
for tid in tqdm(split['test'], desc='Pretrained'):
    blur, sharp = data[tid]
    p, s = img_metrics(blur, sharp)
    res['Blurred input'][tid] = {'psnr': p, 'ssim': s}
    restored['Blurred input'][tid] = blur

    out = restore(model_pre, blur)
    p, s = img_metrics(out, sharp)
    res['Pretrained Restormer'][tid] = {'psnr': p, 'ssim': s}
    restored['Pretrained Restormer'][tid] = out.astype(np.float16)
    save_png16(out, f'{REST_DIR}/pretrained/{tid}.png')

del model_pre
torch.cuda.empty_cache()

for k in ['Blurred input', 'Pretrained Restormer']:
    v = pd.DataFrame(res[k]).T
    print(f'{k:22s}  PSNR {v.psnr.mean():.2f} dB | SSIM {v.ssim.mean():.4f}')

In [ ]:
# Patch must be large enough to contain the blur (kernel radius ~ 2-3 x sigma).
# Level 5 (sigma ~25) used 128 x 4 - kept identical so Level 5 is reproducible.
PATCH_OVERRIDE = None     # e.g. (192, 2) to force a size; set smaller if you hit CUDA out-of-memory
if PATCH_OVERRIDE is not None:
    PATCH, BATCH = PATCH_OVERRIDE
elif BLUR_SIGMA <= 30:
    PATCH, BATCH = 128, 4
elif BLUR_SIGMA <= 50:
    PATCH, BATCH = 192, 2
else:
    PATCH, BATCH = 256, 2

ITERS     = 3000
LR        = 2e-5
VAL_EVERY = 500
TEXT_STD  = 0.05     # reject crops that are only blank paper
print(f'Training patch {PATCH}px x batch {BATCH} | {ITERS} iters | LR {LR}')

FREEZE = ['patch_embed', 'encoder_level1', 'down1_2', 'encoder_level2', 'down2_3', 'encoder_level3', 'down3_4']

model = build_model(PRETRAINED)
if INIT_WEIGHTS != PRETRAINED:
    model.load_state_dict(torch.load(INIT_WEIGHTS, map_location='cpu'))
for name, p in model.named_parameters():
    p.requires_grad = not any(name.startswith(f + '.') for f in FREEZE)

trainable = [p for p in model.parameters() if p.requires_grad]
n_train = sum(p.numel() for p in trainable)
n_frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f'Frozen {n_frozen / 1e6:.2f}M | Trainable {n_train / 1e6:.2f}M')

class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-3):
        super().__init__()
        self.eps = eps
    def forward(self, x, y):
        return torch.mean(torch.sqrt((x - y) ** 2 + self.eps ** 2))

criterion = CharbonnierLoss()
optimizer = torch.optim.AdamW(trainable, lr=LR, weight_decay=1e-4, betas=(0.9, 0.999))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=ITERS, eta_min=1e-7)
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

train_ids = split['train']

def random_patch():
    for attempt in range(20):
        pid = random.choice(train_ids)
        blur, sharp = data[pid]
        H, W = sharp.shape
        y = random.randint(0, H - PATCH)
        x = random.randint(0, W - PATCH)
        g = sharp[y:y + PATCH, x:x + PATCH]
        if g.astype(np.float32).std() > TEXT_STD:
            break
    b = blur[y:y + PATCH, x:x + PATCH]
    k = random.randint(0, 3)
    b = np.rot90(b, k)
    g = np.rot90(g, k)
    if random.random() < 0.5:
        b = np.fliplr(b)
        g = np.fliplr(g)
    return b.copy(), g.copy()

def make_batch():
    bs, gs = [], []
    for _ in range(BATCH):
        b, g = random_patch()
        bs.append(b)
        gs.append(g)
    b = torch.from_numpy(np.stack(bs).astype(np.float32))[:, None].repeat(1, 3, 1, 1).to(DEVICE)
    g = torch.from_numpy(np.stack(gs).astype(np.float32))[:, None].repeat(1, 3, 1, 1).to(DEVICE)
    return b, g

def validate(m):
    ps, ss = [], []
    for vid in split['val']:
        blur, sharp = data[vid]
        p, s = img_metrics(restore(m, blur), sharp)
        ps.append(p)
        ss.append(s)
    m.train()
    return float(np.mean(ps)), float(np.mean(ss))

print('Setup ready')

In [ ]:
# Always save under the clean names (BEST/LAST may point at old 'checkpoints__' files for Level 5)
BEST_OUT = f'{CKPT_DIR}/level{LEVEL}_best.pth'
LAST_OUT = f'{CKPT_DIR}/level{LEVEL}_last.pth'
LOG_TXT  = f'{LOG_DIR}/train_log.txt'

def log(msg):
    print(msg)
    with open(LOG_TXT, 'a') as fp:
        fp.write(f'[{datetime.now():%Y-%m-%d %H:%M:%S}] {msg}\n')

start_iter, best_ssim = 0, -1.0
hist = {'iter': [], 'loss': [], 'val_iter': [], 'val_psnr': [], 'val_ssim': []}

if RESUME and os.path.exists(LAST):
    st = torch.load(LAST, map_location='cpu', weights_only=False)
    model.load_state_dict(st['model'])
    optimizer.load_state_dict(st['opt'])
    scheduler.load_state_dict(st['sched'])
    scaler.load_state_dict(st['scaler'])
    start_iter, best_ssim, hist = st['iter'], st['best_ssim'], st['hist']
    log(f'Level {LEVEL}: resumed from iter {start_iter} (best val SSIM {best_ssim:.4f})')
else:
    log(f'Level {LEVEL}: new run | init {INIT_FROM} | sigma {BLUR_SIGMA}px | patch {PATCH}x{BATCH} | iters {ITERS} | LR {LR}')
    p0, s0 = validate(model)
    hist['val_iter'].append(0)
    hist['val_psnr'].append(p0)
    hist['val_ssim'].append(s0)
    log(f'Iter 0 (start) val: PSNR {p0:.2f} | SSIM {s0:.4f}')

model.train()
running = 0.0
pbar = tqdm(range(start_iter, ITERS), initial=start_iter, total=ITERS)
for it in pbar:
    b, g = make_batch()
    with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
        out = model(b)
    loss = criterion(out.float(), g)

    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(trainable, 1.0)
    scaler.step(optimizer)
    scaler.update()
    scheduler.step()

    running += loss.item()
    step = it + 1
    if step % 50 == 0:
        hist['iter'].append(step)
        hist['loss'].append(running / 50)
        pbar.set_postfix(loss=f'{running / 50:.4f}', lr=f'{scheduler.get_last_lr()[0]:.2e}')
        running = 0.0

    if step % VAL_EVERY == 0 or step == ITERS:
        vp, vs = validate(model)
        hist['val_iter'].append(step)
        hist['val_psnr'].append(vp)
        hist['val_ssim'].append(vs)
        tag = ''
        if vs > best_ssim:
            best_ssim = vs
            torch.save(model.state_dict(), BEST_OUT)
            BEST = BEST_OUT
            tag = '  <- best, saved'
        log(f'Iter {step}: val PSNR {vp:.2f} dB | SSIM {vs:.4f}{tag}')
        torch.save({'model': model.state_dict(), 'opt': optimizer.state_dict(),
                    'sched': scheduler.state_dict(), 'scaler': scaler.state_dict(),
                    'iter': step, 'best_ssim': best_ssim, 'hist': hist}, LAST_OUT)
        LAST = LAST_OUT

json.dump(hist, open(f'{LOG_DIR}/training_history.json', 'w'))
log(f'Training done. Best val SSIM: {best_ssim:.4f}')

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(hist['iter'], hist['loss'])
ax[0].set_title(f'Level {LEVEL} - Charbonnier loss (avg / 50 iters)')
ax[0].set_xlabel('iteration')
ax[1].plot(hist['val_iter'], hist['val_psnr'], marker='o')
ax[1].set_title('Validation PSNR (dB)')
ax[1].set_xlabel('iteration')
ax[2].plot(hist['val_iter'], hist['val_ssim'], marker='o', color='green')
ax[2].set_title('Validation SSIM')
ax[2].set_xlabel('iteration')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/training_curves.png', dpi=150)
plt.show()

In [ ]:
model_ft = build_model(PRETRAINED)
model_ft.load_state_dict(torch.load(BEST, map_location='cpu'))
print('Evaluating:', BEST)
os.makedirs(f'{REST_DIR}/finetuned', exist_ok=True)

for tid in tqdm(split['test'], desc='Fine-tuned'):
    blur, sharp = data[tid]
    out = restore(model_ft, blur)
    p, s = img_metrics(out, sharp)
    res['Fine-tuned Restormer'][tid] = {'psnr': p, 'ssim': s}
    restored['Fine-tuned Restormer'][tid] = out.astype(np.float16)
    save_png16(out, f'{REST_DIR}/finetuned/{tid}.png')

del model_ft
torch.cuda.empty_cache()

for k in res:
    v = pd.DataFrame(res[k]).T
    print(f'{k:22s}  PSNR {v.psnr.mean():.2f} dB | SSIM {v.ssim.mean():.4f}')

In [ ]:
import easyocr
reader = easyocr.Reader(['en'], gpu=True)

def to_u8(img):
    return (np.clip(img.astype(np.float32), 0, 1) * 255).round().astype(np.uint8)

def ocr_text(img):
    out = reader.readtext(to_u8(img), detail=1, paragraph=False)
    if len(out) == 0:
        return ''
    items = []
    for box, txt, conf in out:
        ys = [pt[1] for pt in box]
        xs = [pt[0] for pt in box]
        items.append((np.mean(ys), min(xs), max(ys) - min(ys), txt))
    items.sort(key=lambda t: t[0])
    lines = []
    for itm in items:
        if len(lines) > 0 and abs(itm[0] - lines[-1][0][0]) < 0.5 * itm[2]:
            lines[-1].append(itm)
        else:
            lines.append([itm])
    text_lines = []
    for ln in lines:
        ln.sort(key=lambda t: t[1])
        text_lines.append(' '.join(t[3] for t in ln))
    return ' '.join(text_lines)

def clean(s):
    return ' '.join(s.split())

def char_acc(pred, gt):
    gt_c = clean(gt)
    if len(gt_c) == 0:
        return float('nan')
    d = Levenshtein.distance(clean(pred), gt_c)
    return max(0.0, 1.0 - d / len(gt_c))

def token_acc(pred, gt):
    gt_tok = clean(gt).split()
    if len(gt_tok) == 0:
        return float('nan')
    pred_tok = clean(pred).split()
    hits = 0
    for t in gt_tok:
        if t in pred_tok:
            pred_tok.remove(t)
            hits += 1
    return hits / len(gt_tok)

res['Sharp GT (upper bound)'] = {}
restored['Sharp GT (upper bound)'] = {tid: data[tid][1] for tid in split['test']}

ocr_rows = []
for method in ['Sharp GT (upper bound)', 'Blurred input', 'Pretrained Restormer', 'Fine-tuned Restormer']:
    for tid in tqdm(split['test'], desc=method):
        pred = ocr_text(restored[method][tid])
        gt = gt_text[tid]
        if tid not in res[method]:
            res[method][tid] = {}
        res[method][tid]['char_acc'] = char_acc(pred, gt)
        res[method][tid]['token_acc'] = token_acc(pred, gt)
        ocr_rows.append({'method': method, 'id': tid, 'gt': clean(gt), 'ocr': clean(pred)})

pd.DataFrame(ocr_rows).to_csv(f'{OUT}/ocr_outputs.csv', index=False)
print('OCR done')

In [ ]:
pair_by_id = {p['id']: p for p in pairs}
rows = []
for method, d in res.items():
    for tid, m in d.items():
        rows.append({'method': method, 'id': tid, 'font': pair_by_id[tid]['font'], **m})
df = pd.DataFrame(rows)
df.to_csv(f'{OUT}/level{LEVEL}_per_image.csv', index=False)

order = ['Blurred input', 'Pretrained Restormer', 'Fine-tuned Restormer', 'Sharp GT (upper bound)']
summary = df.groupby('method')[['psnr', 'ssim', 'char_acc', 'token_acc']].mean().reindex(order)
summary.to_csv(f'{OUT}/level{LEVEL}_summary.csv')
by_font = df.groupby(['method', 'font'])[['psnr', 'ssim', 'char_acc']].mean().reindex(order, level=0)
by_font.to_csv(f'{OUT}/level{LEVEL}_by_font.csv')

print(f'=== LEVEL {LEVEL} - TEST SET (n = {len(split["test"])}) ===')
print(summary.round(4).to_string())
print('\n=== By font ===')
print(by_font.round(4).to_string())

ft = summary.loc['Fine-tuned Restormer']
pre = summary.loc['Pretrained Restormer']
bl = summary.loc['Blurred input']
print('\n=== Gains of fine-tuned model ===')
print(f"vs pretrained : PSNR {ft.psnr - pre.psnr:+.2f} dB | SSIM {ft.ssim - pre.ssim:+.4f} | CharAcc {100 * (ft.char_acc - pre.char_acc):+.1f} pts")
print(f"vs blurred    : PSNR {ft.psnr - bl.psnr:+.2f} dB | SSIM {ft.ssim - bl.ssim:+.4f} | CharAcc {100 * (ft.char_acc - bl.char_acc):+.1f} pts")

In [ ]:
n_test = len(split['test'])
show_ids = [split['test'][0], split['test'][n_test // 2], split['test'][-1]]   # mix of both fonts
cols = ['Blurred input', 'Pretrained Restormer', 'Fine-tuned Restormer', 'Sharp GT (upper bound)']

fig, axes = plt.subplots(len(show_ids), 4, figsize=(20, 3.2 * len(show_ids)))
for r, tid in enumerate(show_ids):
    H, W = data[tid][1].shape
    y0, x0 = H // 2 - 200, W // 2 - 600     # central text crop
    for c, method in enumerate(cols):
        crop = restored[method][tid][y0:y0 + 400, x0:x0 + 1200].astype(np.float32)
        axes[r, c].imshow(crop, cmap='gray', vmin=0, vmax=1)
        axes[r, c].axis('off')
        title = method
        if 'ssim' in res[method].get(tid, {}):
            title += f"\nPSNR {res[method][tid]['psnr']:.2f} | SSIM {res[method][tid]['ssim']:.3f}"
        if r == 0 or 'ssim' in res[method].get(tid, {}):
            axes[r, c].set_title(title, fontsize=10)
    axes[r, 0].set_ylabel(tid)
plt.suptitle(f'Level {LEVEL}', fontsize=14)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/level{LEVEL}_comparison.png', dpi=150)
plt.show()

In [ ]:
# ---- update this level's manifest.json ----
manifest_path = f'{LEVEL_DIR}/manifest.json'
if os.path.exists(manifest_path):
    manifest = json.load(open(manifest_path))
else:
    manifest = {'level': LEVEL}

def rel(p):
    return os.path.relpath(p, LEVEL_DIR)

manifest.update({
    'status': 'trained',
    'updated': datetime.now().strftime('%Y-%m-%d %H:%M'),
    'pretrained_base': f'../pretrained/{os.path.basename(PRETRAINED)}',
    'init_from': INIT_FROM,
    'init_weights': INIT_WEIGHTS.replace(PHASE2 + '/', ''),
    'dataset': rel(ZIP_PATH) if os.path.exists(ZIP_PATH) else rel(DATA_DIR),
    'checkpoints': {'best': rel(BEST), 'last': rel(LAST)},
    'results_dir': rel(OUT),
    'blur_sigma_px': BLUR_SIGMA,
    'train_config': {'patch': PATCH, 'batch': BATCH, 'iters': ITERS, 'lr': LR, 'tile': TILE, 'overlap': OVERLAP},
    'split_sizes': {k: len(v) for k, v in split.items()},
    'best_val_ssim': round(best_ssim, 4),
    'test_summary': summary.round(4).reset_index().to_dict(orient='records'),
})
json.dump(manifest, open(manifest_path, 'w'), indent=2)
print('Manifest updated:', manifest_path)
print(f'Level {LEVEL} complete. Results: {OUT}')

## All levels so far
Run this any time — it collects every `levelN_summary.csv` that exists and plots the trend across blur levels.

In [ ]:
all_rows = []
for lv in range(5, 11):
    f = level_result_file(lv, f'level{lv}_summary.csv')
    if f is None:
        continue
    s = pd.read_csv(f)
    s.insert(0, 'level', lv)
    all_rows.append(s)

if len(all_rows) == 0:
    print('No level summaries found yet')
else:
    allsum = pd.concat(all_rows, ignore_index=True)
    allsum.to_csv(f'{PHASE2}/all_levels_summary.csv', index=False)
    print(allsum.round(4).to_string(index=False))

    methods = ['Blurred input', 'Pretrained Restormer', 'Fine-tuned Restormer', 'Sharp GT (upper bound)']
    fig, ax = plt.subplots(1, 3, figsize=(18, 4))
    for i, metric in enumerate(['psnr', 'ssim', 'char_acc']):
        for mth in methods:
            sub = allsum[allsum.method == mth].sort_values('level')
            if sub[metric].notna().any():
                ax[i].plot(sub.level, sub[metric], marker='o', label=mth)
        ax[i].set_title(metric.upper())
        ax[i].set_xlabel('blur level')
        ax[i].set_xticks(sorted(allsum.level.unique()))
    ax[0].legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(f'{PHASE2}/all_levels_trend.png', dpi=150)
    plt.show()
    print('Saved:', f'{PHASE2}/all_levels_summary.csv')